# Modern Dev Tour: GitHub, VS Code, and development environments. And more Pandas 


By the end of today you will be able to:

- Create a GitHub account, a repository, and commit a notebook through the web UI with a commit message that actually says something.
- State, in one sentence each, what VS Code and environment management tools are for, and why Colab doesn't ask you to deal with either :)
- Read a CSV with pandas and inspect it with `.head()`, `.describe()`, `.sort_values()`
- Explain why three pandas lines replace what a manual loop over a CSV's rows would take much longer to write correctly



## Setup

1. Create a new Google Colab notebook
2. Run the setup cell below. As before, nothing in today's pandas section actually needs any of these five packages, but every Part 2 notebook opens with this cell so the pattern stays automatic

In [ ]:
# Run this cell first (takes ~1 minute). Colab already includes geopandas.
%pip install -q libpysal esda splot mapclassify contextily

## GitHub for real

You may have created a GitHub account already (the previous class made time for it). 

If you haven't yet, do it now: go to `github.com`, sign up with a personal email address (not a class email you'll lose access to), and pick a username you'd be comfortable putting on a resume.

 You'll use this account well past this course. (For step-by-step signup instructions, 2FA setup, and the free Student Developer Pack, see the account setup guide.)

**Step 1: Create a repository.** From your GitHub homepage, click the green "New" button (or the `+` in the top-right corner → "New repository"). Give it a name (something like `spatial-programming-notebooks` is fine) leave it Public, and check "Add a README file" so you start with one committed file already in place.

**Step 2: Upload a file.** Inside your new repository, click "Add file" → "Upload files," then drag any file.

**Step 3: Write a commit message that communicates something.** Every upload and every edit on GitHub asks for a commit message before it saves. "update," "fixed stuff," and "asdf" are three of the most common commit messages in the world, and ***all three are useless to you six months from now***. A commit message answers the question: *what changed, and why?*

- Weak: `update notebook`
- Better: `Add first geopandas plot of Portage County streams`

The habit is small, but the habit is ALSO aimed at your future self and anyone who reads your work after you.

**Your turn:** Create the repository described above (or use one you already have), upload any file, and write a commit message describing what that specific file is, not "upload."

## VS Code: a (very) short tour

You won't install VS Code today (and this course doesn't require it), as Colab is enough for everything you'll submit. But you'll see it mentioned *constantly* outside this course.

**What it is:** Visual Studio Code (free, from Microsoft) is a code editor that runs on your own machine rather than in a browser tab. 

**Compared to Colab:**

| Aspect | Colab | VS Code |
|---|---|---|
| Where it runs | Google's servers, in your browser | Your own laptop |
| Setup | None, just open a tab | Install the app, plus a Python interpreter |
| Best for | Quick, shareable, no-install work (this course) | Larger projects, many files, working offline |
| Notebooks? | Native | Supported via an extension, uses same `.ipynb` format |

**What you'd see if you opened it:** a file browser on the left, an editor in the center, and an integrated terminal at the bottom. That's one window for writing code, running it, and managing files, instead of switching between separate apps.

**So why this course doesn't use it?:** every Part 2 notebook you'll write fits easily in one file, run occasionally, shared with an instructor. These are Colab's strengths. VS Code is useful for bigger, longer-lived projects. Recognizing *when* a "heavier" tool is worth the switch is its own skill.

## Pandas, for real: week 1's manual loop in three lines

`ohio_precip_2025.csv` contains precipitation totals for over 400 stations across Ohio.

If I had asked in week 1 for you to find the station with the greatest precip, How would you have accomplished the task?



## you would have had to:

1. open `ohio_precip_2025.csv`
2. loop over every line
3. split each one on commas by hand
4. convert the precipitation field to a number
5. keep a running max as you go

This would include remembering to skip the header row, and to handle a blank field if one turns up. Plausible code, and a fair bit of it.

***So let's take a better approach***

In [ ]:
DATA_BASE = "https://raw.githubusercontent.com/pjbitterman/KSU_x9076/main/data"

# Local path fallback (uncomment if running outside Colab without internet access):
# DATA_BASE = "../data"

import pandas as pd

precip = pd.read_csv(f"{DATA_BASE}/act1/ohio_precip_2025.csv")
precip.head()

`.head()` shows the first five rows without printing all 410. 

This is sufficient to confirm the columns loaded the way you expect (`station_id`, `name`, `lat`, `lon`, `precip_mm_2025`)

In [ ]:
precip.describe()

`.describe()` is a useful next step. Build this habit.



With a single function, `pandas` calculates count, mean, standard deviation, min, max, and quartiles for every numeric column. 

The `count` at 410 confirms every station is in the DataFrame, and no rows were dropped silently.

### A common question 

Imagine a lab/practicum/sketch question like....

***Which five stations were the wettest, and by how much? pandas can answer it in one line, but it requires chaining function calls together.***

### Let's go step by step first

**Step 1: sort.** `.sort_values()` takes a column name and returns a new DataFrame with the rows reordered by that column. The original `precip` is not changed.

In [ ]:
sorted_precip = precip.sort_values("precip_mm_2025")
sorted_precip

All 410 rows present, just reordered. 

The default order is ascending, so the driest station is on top and the wettest is at the bottom. That is backwards for this question.

**Step 2: flip the order.** `ascending=False` puts the largest values first.

In [ ]:
wettest_first = precip.sort_values("precip_mm_2025", ascending=False)
wettest_first.head()

Same 410 rows, now with the wettest station on top. `.head()` here is only included so that we can check our work. It's not required, and it doesn't alter the data

**Step 3: keep only the first five rows.** `.head(5)` returns a new, smaller DataFrame: 

the top five rows of whatever table it was called on. 

Let's store it this time, because it is the result we're looking for

In [ ]:
top_five = wettest_first.head(5)
top_five

Five rows, but still every column, including `station_id`, `lat`, and `lon`, which the question did not ask for.

**Step 4: keep only the columns you want.** The double-bracket form selects several columns by name and returns a narrower table.

In [ ]:
top_five[["name", "precip_mm_2025"]]

That is the answer we're looking for: five stations, two columns.

**Now let's accomplish the same thing in one line.** 

Review the four steps. 

Each step called a method on a DataFrame and got a DataFrame back, and each was called on the result of the step before it. 

That means the three intermediate names (`sorted_precip`, `wettest_first`, `top_five`) were only ever handed from one step to the next.

If we drop them, the four steps join into a single chain. Read it left to right: sort, flip, take five, retain two columns.

In [ ]:
precip.sort_values("precip_mm_2025", ascending=False).head(5)[["name", "precip_mm_2025"]]

Compared to the start of the semester:

- `.sort_values()` with `.head(5)` replaces the whole manual loop 
- no running-maximum variable to initialize and keep track of
- no counter to stop after five
- no risk of comparing the header row by mistake

The same line, flipped, answers the opposite question (which stations had the *least* precipitation) just as quickly:

In [ ]:
precip.sort_values("precip_mm_2025").head(5)[["name", "precip_mm_2025"]]

**Your turn:** 

Using the same three ideas — read (already done), sort, look — find the 5 stations **closest to** the statewide mean precipitation, rather than the extremes. 

**HINT: You likely need to create a column that doesn't exist yet!**

Making a new column uses the same bracket syntax as reading one, but on the left side of an `=`:

```python
precip["new_column"] = <something computed from existing columns>
```

Whatever is on the right side is computed for every row at once - it doesn't require a loop. 

Arithmetic on a column applies to ***each*** value in it, so `precip["precip_mm_2025"] - precip["precip_mm_2025"].mean()` creates a column of differences, one per station. 

Note, this doesn't answer everything you need - there's at least one more step.


## Wrap-up

**If there's time**

1. Calculate the standard deviation of `precip_mm_2025` two ways: once via `.describe()` (already visible above) and once by hand. Confirm your manual result matches (within rounding).
2. Sort the DataFrame two ways in one cell (ascending and descending) and print just the `name` column for each, seven values for each.


**Next time:** pandas' spatial cousin, GeoPandas